### 🧠 Answer Synthesis from Multiple Sources
✅ What Is It?

Answer synthesis from multiple sources is the process where an AI agent collects information from different retrieval tools or knowledge bases, and merges that information into a single, coherent, and contextually rich answer.

This is a core capability in Agentic RAG, where the system is more than just a simple retriever — it plans, retrieves, and then synthesizes an answer that draws from multiple sources.


🎯 Why It’s Needed
Most real-world queries are:
- Multifaceted (require multiple types of information)
- Ambiguous or incomplete (need refinement)
- Open-ended (don’t map to a single document or source)

🔍 This makes retrieving from a single vector DB insufficient.

Instead, we want an agent that can:

- Decide what to fetch from where (retrieval planning)
- Retrieve content from multiple tools (e.g., Wikipedia, PDFs, APIs, SQL)
- Evaluate and merge that context
- Produce a single human-like response

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')

# LLM Model
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
llm

c:\Users\heman\Desktop\05 Ultimate RAG Bootcamp Using Langchain,LangGraph and Langsmith by Krish Naik\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x000001F1EA31BA10>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001F1EA7B0590>, root_client=<openai.OpenAI object at 0x000001F1EA3197F0>, root_async_client=<openai.AsyncOpenAI object at 0x000001F1EA7B02F0>, model_name='gpt-4o', model_kwargs={}, openai_api_key=SecretStr('**********'))

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.schema import Document

from langchain.tools import WikipediaQueryRun
from langchain.utilities import WikipediaAPIWrapper

from langchain_community.document_loaders import ArxivLoader


# load text docs
def load_text_retriever(file_path):
    docs = TextLoader(file_path, encoding="utf-8").load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)
    vs = FAISS.from_documents(chunks, OpenAIEmbeddings())
    
    return vs.as_retriever()

# load YouTube transcript
def load_youtube_retriever():
    # Mocked YouTube transcript text
    content = """
    This video explains how agentic AI systems rely on feedback loops, memory, and tool use.
    It compares them to traditional pipeline-based LLMs. Temporal reasoning and autonomous tasking are emphasized.
    """
    doc = Document(page_content=content, metadata={"source": "youtube"})
    vectorstore = FAISS.from_documents([doc], OpenAIEmbeddings())
    return vectorstore.as_retriever()


# Wikipedia search
def wikipedia_search(query: str) -> str:
    print("🌐 Searching Wikipedia...")
    return WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())(query)

# arxiv search
def arxiv_search(query: str) -> str:
    print("📄 Searching ArXiv...")
    results = ArxivLoader(query).load()
    return "\n\n".join(doc.page_content for doc in results[:2]) or "No relevant papers found."

In [3]:
text_retriever = load_text_retriever("research.txt")
youtube_retriever = load_youtube_retriever()

In [4]:
from typing import List
from pydantic import BaseModel
from langchain.schema import Document

# -------------------------------
# 2. LangGraph State Definition
# -------------------------------

class MultiSourceRAGState(BaseModel):
    question: str
    text_docs: List[Document] = []
    yt_docs: List[Document] = []
    wiki_context: str = ""
    arxiv_context: str = ""
    final_answer: str = ""

In [5]:
# -------------------------------
# 3. Nodes
# -------------------------------

def retrieve_text(state: MultiSourceRAGState) -> MultiSourceRAGState:
    docs = text_retriever.invoke(state.question)
    return state.model_copy(update={"text_docs": docs})

def retrieve_yt(state: MultiSourceRAGState) -> MultiSourceRAGState:
    docs = youtube_retriever.invoke(state.question)
    return state.model_copy(update={"yt_docs": docs})

def retrieve_wikipedia(state: MultiSourceRAGState) -> MultiSourceRAGState:
    result = wikipedia_search(state.question)
    return state.model_copy(update={"wiki_context": result})

def retrieve_arxiv(state: MultiSourceRAGState) -> MultiSourceRAGState:
    result = arxiv_search(state.question)
    return state.model_copy(update={"arxiv_context": result})

In [6]:
## synthesize
def synthesize_answer(state: MultiSourceRAGState) -> MultiSourceRAGState:
    
    context = ""

    context += "\n\n[Internal Docs]\n" + "\n".join([doc.page_content for doc in state.text_docs])
    context += "\n\n[YouTube Transcript]\n" + "\n".join([doc.page_content for doc in state.yt_docs])
    context += "\n\n[Wikipedia]\n" + state.wiki_context
    context += "\n\n[ArXiv]\n" + state.arxiv_context

    prompt = f"""You have retrieved relevant context from multiple sources. Now synthesize a complete and coherent answer.

Question: {state.question}
Context: {context}

Final Answer:
"""

    answer = llm.invoke(prompt).content.strip()
    return state.model_copy(update={"final_answer": answer})

In [7]:
from langgraph.graph import StateGraph,START, END
# -------------------------------
# 4. LangGraph Graph
# -------------------------------

# Build the graph
graph = StateGraph(MultiSourceRAGState)

# Nodes
graph.add_node("retrieve_text", retrieve_text)
graph.add_node("retrieve_yt", retrieve_yt)
graph.add_node("retrieve_wiki", retrieve_wikipedia)
graph.add_node("retrieve_arxiv", retrieve_arxiv)
graph.add_node("synthesize", synthesize_answer)

# Edges
graph.add_edge(START, "retrieve_text")
graph.add_edge("retrieve_text", "retrieve_yt")
graph.add_edge("retrieve_yt", "retrieve_wiki")
graph.add_edge("retrieve_wiki", "retrieve_arxiv")
graph.add_edge("retrieve_arxiv", "synthesize")
graph.add_edge("synthesize", END)

# Compile
workflow = graph.compile()

In [8]:
# -------------------------------
# 5. Run RAG Agent
# -------------------------------

question = "What is Machine Learning in short?"
state = MultiSourceRAGState(question=question)
result = workflow.invoke(state)

print("✅ Final Answer:\n")
print(result["final_answer"])

🌐 Searching Wikipedia...


C:\Users\heman\AppData\Local\Temp\ipykernel_13176\2096244237.py:37: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  return WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())(query)


📄 Searching ArXiv...
✅ Final Answer:

Machine learning (ML) is a subfield of artificial intelligence that focuses on the development and study of algorithms capable of learning from data and generalizing to new, unseen data, thus enabling tasks to be performed without explicit instructions. At its core, machine learning leverages statistical models to discover patterns or insights in data, enabling systems to improve their performance on a specific set of tasks through experience. Notable advancements, particularly in deep learning with neural networks, have significantly enhanced the capabilities of machine learning, surpassing many traditional approaches in fields such as natural language processing, computer vision, and speech recognition. Essentially, ML systems are trained to recognize patterns and make decisions with minimal human intervention, employing techniques from statistics, linear algebra, and optimization to find solutions and predict outcomes.


In [9]:
result

{'question': 'What is Machine Learning in short?',
 'text_docs': [Document(id='b1f84d0b-5af0-4de0-a920-e795300e9bf9', metadata={'source': 'research.txt'}, page_content='LLaMA2:\n- Instruction-tuned with 20k internal support tickets\n- Combined with RAG for chatbot Q&A\n- Latency: ~300ms on A100\n- Evaluation metric: Win rate vs human (75%)\n- GPT-4 judge feedback: “Contextual reasoning good, factuality needs work”\n- Prompts evaluated: Alpaca format, ChatML, and custom XML-based memory tags'),
  Document(id='dd6eacd5-9ff9-4d29-9e93-2c9af27e1c05', metadata={'source': 'research.txt'}, page_content='3. Tool-augmented prompting:\n   - Integrated LangGraph + Wikipedia + SQL search\n   - Enables dynamic retrieval-agent reasoning\n   - Example: "Give me customer insights from SQL, verify with wiki"\n\n4. Human evaluation protocol:\n   - Internal annotators score fluency, helpfulness, correctness\n   - GPT-4 also used as synthetic evaluator'),
  Document(id='acffd837-7a7c-44ae-b559-ea66226db4c